In [ ]:
# Install Tiny-SIMA into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("Source path added to kernel:", source_root)
print("CUDA detected:", has_cuda)


# Phase 3: train a goal-conditioned executor

This notebook trains one recurrent PPO policy across all six nearby-target skills. The training environment assigns a requested categorical skill to each environment slot, and that skill ID is embedded and supplied to the shared policy at every action step. There is no VLM planner or learned skill selection in this phase.

Start with the debug configuration. Once it runs successfully, set `RUN_1M = True` for the longer GPU experiment.

In [ ]:
import jax
import matplotlib.pyplot as plt

from tiny_sima.config import load_skill_config
from tiny_sima.evaluation.goal_conditioned import (
    diagnose_goal_conditioned_checkpoint,
    evaluate_goal_conditioned_checkpoint,
)
from tiny_sima.executor.skill_ppo import train_goal_conditioned

print("JAX devices:", jax.devices())


In [ ]:
RUN_1M = False

config_name = "skill_ppo_1m.yaml" if RUN_1M else "skill_ppo_debug.yaml"
run_name = "skill-ppo-1m" if RUN_1M else "skill-ppo-debug"
config = load_skill_config(repo_root / "configs" / config_name).with_overrides(
    output_dir=str(repo_root / "runs" / run_name),
)
config.to_dict()


In [ ]:
summary = train_goal_conditioned(config)
summary


In [ ]:
deterministic_results = evaluate_goal_conditioned_checkpoint(
    summary["checkpoint"],
    num_episodes_per_skill=64,
    seed=100,
)
stochastic_results = evaluate_goal_conditioned_checkpoint(
    summary["checkpoint"],
    num_episodes_per_skill=64,
    seed=200,
    deterministic=False,
)

for deterministic, stochastic in zip(
    deterministic_results,
    stochastic_results,
    strict=True,
):
    print(
        f"{deterministic['skill_id']:<22} "
        f"det={deterministic['success_rate']:.3f} "
        f"stoch={stochastic['success_rate']:.3f} "
        f"timeout={deterministic['feasible_timeout_rate']:.3f} "
        f"precondition_lost={deterministic['precondition_lost_rate']:.3f}"
    )


In [ ]:
diagnostics = diagnose_goal_conditioned_checkpoint(
    summary["checkpoint"],
    num_scenarios_per_skill=256,
    seed=300,
)

for diagnostic in diagnostics:
    initial = diagnostic["initial"]
    followup = diagnostic["after_oracle_turn"]
    if followup is None:
        print(
            f"{diagnostic['skill_id']:<22} "
            f"direct_acc={initial['top1_accuracy']:.3f} "
            f"direct_p={initial['mean_target_probability']:.3f} "
            f"entropy={initial['mean_action_entropy']:.3f}"
        )
    else:
        print(
            f"{diagnostic['skill_id']:<22} "
            f"turn_acc={initial['top1_accuracy']:.3f} "
            f"turn_p={initial['mean_target_probability']:.3f} "
            f"do_acc={followup['top1_accuracy']:.3f} "
            f"do_p={followup['mean_target_probability']:.3f} "
            f"sequence={diagnostic['deterministic_sequence_accuracy']:.3f} "
            f"entropy=({initial['mean_action_entropy']:.3f}, "
            f"{followup['mean_action_entropy']:.3f})"
        )


In [ ]:
skill_names = [result["skill_id"] for result in deterministic_results]
deterministic_rates = [
    result["success_rate"] for result in deterministic_results
]
stochastic_rates = [
    result["success_rate"] for result in stochastic_results
]
positions = list(range(len(skill_names)))

plt.figure(figsize=(10, 4))
plt.bar(
    [position - 0.2 for position in positions],
    deterministic_rates,
    width=0.4,
    label="deterministic",
)
plt.bar(
    [position + 0.2 for position in positions],
    stochastic_rates,
    width=0.4,
    label="stochastic",
)
plt.ylim(0.0, 1.0)
plt.ylabel("Success rate")
plt.title("Goal-conditioned executor by requested skill")
plt.xticks(positions, skill_names, rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.show()
